<이진분류 : 데이터를 두가지 클래스(0 or 1, 참 or 거짓)로 분류하는 문제>

## 1 문제정의
- 미국의 인구조사 데이터(1994)를 바탕으로 만들어진 데이터
- 이 데이터에서 각 사람의 소득을 예측하려는 것이 목적
- 나이, 결혼여부, 직종 등의 컬럼이 있음
- income : 연소득 $50,000 초과 or 이하로 구분 -> 이진분류 
- 평가기준 : ROC-AUC
- 제출파일은 예측값만 result.csv 파일로 생성

문제풀이
- 사람이 가진 다양한 정보(독립변수)를 보고 연봉이 $50,000 이상인지 이하인지를 예측하는 문제
- 입력(독립변수) : 나이, 결혼여부, 직종, 학력, 근무시간 등 
- 출력(종속변수) : 소득이 >50K or <=50K 인지 

In [1]:
# 판다스 라이브러리
import pandas as pd

# 데이터 불러오기
# 아래 링크로 들어가서 메모장으로 붙여넣어서 파일 생성
train = pd.read_csv('DataSet/train.csv')
test = pd.read_csv('DataSet/test.csv')

train

,id,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,3331,34.0,State-gov,177331,Some-college,10,Married-civ-spouse,Prof-specialty,Husband,Black,Male,4386,0,40.0,United-States,>50K
1,19749,58.0,Private,290661,HS-grad,9,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,40.0,United-States,<=50K
2,1157,48.0,Private,125933,Some-college,10,Widowed,Exec-managerial,Unmarried,Black,Female,0,1669,38.0,United-States,<=50K
3,693,58.0,Private,100313,Some-college,10,Married-civ-spouse,Protective-serv,Husband,White,Male,0,1902,40.0,United-States,>50K
4,12522,41.0,Private,195661,Some-college,10,Married-civ-spouse,Transport-moving,Husband,White,Male,0,0,54.0,United-States,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29299,15999,28.0,Private,47168,10th,6,Never-married,Machine-op-inspct,Own-child,White,Female,0,0,40.0,United-States,<=50K
29300,21604,44.0,Local-gov,231793,Doctorate,16,Married-spouse-absent,Prof-specialty,Unmarried,White,Female,0,0,38.0,United-States,<=50K
29301,26839,41.0,Local-gov,201435,HS-grad,9,Married-civ-spouse,Transport-moving,Husband,Black,Male,0,0,40.0,United-States,<=50K
29302,16681,43.0,Private,137722,HS-grad,9,Married-civ-spouse,Handlers-cleaners,Husband,White,Male,0,0,40.0,United-States,<=50K


## 2 탐색적 데이터 분석(EDA)

In [2]:
# 데이터 샘플
train.head(3)
# 데이터 크기
train.shape, test.shape
# 자료형 (타입)
train.info()
# 수치형 컬럼 통계값 확인(train)
train.describe()
# 수치형 컬럼 통계값 확인(test)
test.describe()
# 범주형 컬럼 통계값 확인 (train)
train.describe(include='O')
# 범주형 컬럼 통계값 확인 (test)
test.describe(include='object')
# 결측치 (train)
train.isnull().sum()
# 결측치 (test)
test.isnull().sum()
# label(target)별 개수 확인
train['income'].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29304 entries, 0 to 29303
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              29304 non-null  int64  
 1   age             29292 non-null  float64
 2   workclass       27642 non-null  object 
 3   fnlwgt          29304 non-null  int64  
 4   education       29304 non-null  object 
 5   education.num   29304 non-null  int64  
 6   marital.status  29304 non-null  object 
 7   occupation      27636 non-null  object 
 8   relationship    29304 non-null  object 
 9   race            29304 non-null  object 
 10  sex             29304 non-null  object 
 11  capital.gain    29304 non-null  int64  
 12  capital.loss    29304 non-null  int64  
 13  hours.per.week  29291 non-null  float64
 14  native.country  28767 non-null  object 
 15  income          29304 non-null  object 
dtypes: float64(2), int64(5), object(9)
memory usage: 3.6+ MB


income
<=50K    22263
>50K      7041
Name: count, dtype: int64

## 3 데이터 전처리

In [3]:
# 결측치 확인
# 각 컬럼별 결측치 개수
print(train.isnull().sum())
print()
# 전체 결측치 합계
print(train.isnull().sum().sum())
print()
# 전체행 대비 결측치 비율 - 비어있는게 많은 값은 어떻게 하면 채워넣을지 생각
print((train.isnull().sum()/len(train)))
print()
# 결측치 있는 컬럼만 출력
print(train.columns[train.isnull().any()])

id                   0
age                 12
workclass         1662
fnlwgt               0
education            0
education.num        0
marital.status       0
occupation        1668
relationship         0
race                 0
sex                  0
capital.gain         0
capital.loss         0
hours.per.week      13
native.country     537
income               0
dtype: int64

3892

id                0.000000
age               0.000410
workclass         0.056716
fnlwgt            0.000000
education         0.000000
education.num     0.000000
marital.status    0.000000
occupation        0.056921
relationship      0.000000
race              0.000000
sex               0.000000
capital.gain      0.000000
capital.loss      0.000000
hours.per.week    0.000444
native.country    0.018325
income            0.000000
dtype: float64

Index(['age', 'workclass', 'occupation', 'hours.per.week', 'native.country'], dtype='object')


In [4]:
# 결측치가 있는 데이터(행) 전체 삭제 및 확인 dropna() / 기본값 axis=0
print("처리전:", train.shape)
df = train.dropna()
print("처리후:", df.shape)

# 결측치가 있는 특정컬럼 데이터(행)삭제
df = train.dropna(subset=['native.country','workclass'])
df.isnull().sum()

처리전: (29304, 16)
처리후: (27096, 16)


id                 0
age               12
workclass          0
fnlwgt             0
education          0
education.num      0
marital.status     0
occupation         6
relationship       0
race               0
sex                0
capital.gain       0
capital.loss       0
hours.per.week    13
native.country     0
income             0
dtype: int64

In [5]:
# 결측치가 있는 컬럼 삭제 dropna(axis=1)
print("처리전:", train.shape)
df = train.dropna(axis=1)
print("처리후:", df.shape)

# 결측치가 많은 특정 컬럼 삭제 drop
print("처리전:", train.shape)
df = train.drop(['native.country','workclass'], axis=1)
print("처리후:", df.shape)

처리전: (29304, 16)
처리후: (29304, 11)
처리전: (29304, 16)
처리후: (29304, 14)


### 결측치 채우기

In [6]:
# 결측치 채우기(최빈값)
train.describe(include="O")

m = train['workclass'].mode()[0] # Private
train['workclass'] = train['workclass'].fillna(m) # workclass 열의 결측치를 Private로 채우기
train.isnull().sum() # workclass 결측치 확인

m = train['native.country'].mode()[0] # United-States
train['native.country'] = train['native.country'].fillna(m)
train.isnull().sum()

id                   0
age                 12
workclass            0
fnlwgt               0
education            0
education.num        0
marital.status       0
occupation        1668
relationship         0
race                 0
sex                  0
capital.gain         0
capital.loss         0
hours.per.week      13
native.country       0
income               0
dtype: int64

In [7]:
# 결측치 채우기(새로운 카테고리(X)로 분류)
train['occupation'] = train['occupation'].fillna('X')
train['occupation'].value_counts()  # X 값이 1668개 생긴 것 확인

occupation
Prof-specialty       3683
Craft-repair         3677
Exec-managerial      3632
Adm-clerical         3411
Sales                3295
Other-service        2993
Machine-op-inspct    1822
X                    1668
Transport-moving     1434
Handlers-cleaners    1223
Farming-fishing       891
Tech-support          842
Protective-serv       590
Priv-house-serv       135
Armed-Forces            8
Name: count, dtype: int64

In [8]:
# 결측치 채우기(수치형 - 평균값)
value = int(train['age'].mean()) # int로 해도 train.info 확인시 int형으로 변경되지 X
value
train['age'] = train['age'].fillna(value)
train.isnull().sum()
# train.info()

id                 0
age                0
workclass          0
fnlwgt             0
education          0
education.num      0
marital.status     0
occupation         0
relationship       0
race               0
sex                0
capital.gain       0
capital.loss       0
hours.per.week    13
native.country     0
income             0
dtype: int64

In [9]:
# 결측치 채우기(수치형 - 중앙값)
value = int(train['hours.per.week'].median())
value
train['hours.per.week'] = train['hours.per.week'].fillna(value)
train.isnull().sum()

id                0
age               0
workclass         0
fnlwgt            0
education         0
education.num     0
marital.status    0
occupation        0
relationship      0
race              0
sex               0
capital.gain      0
capital.loss      0
hours.per.week    0
native.country    0
income            0
dtype: int64

In [10]:
# test 데이터 결측치 처리(train에서 했던 것과 동일하게)
test.isnull().sum() # age, workclass, occupation, hours.per.week, native.country 결측치 존재 확인

# age - 평균값으로
test['age'] = test['age'].fillna(int(test['age'].mean()))
# test.isnull().sum()

# workclass - 최빈값으로
test['workclass'] = test['workclass'].fillna(test['workclass'].mode()[0])
# test['workclass'].mode()[0] # Private 확인
# test.isnull().sum() 확인 작업하면서 코드 재실행

# occupation - 새로운 카테고리 X로
test['occupation'] = test['occupation'].fillna('X')
test.isnull().sum()
# test['occupation'].value_counts() # X 데이터 결측치(175)만큼 생성

# hours.per.week - 중앙값으로
test['hours.per.week'] = test['hours.per.week'].fillna(int(test['hours.per.week'].median()))
# test.isnull().sum()

# native.country - 최빈값
test['native.country'] = test['native.country'].fillna(test['native.country'].mode()[0])
test.isnull().sum()

id                0
age               0
workclass         0
fnlwgt            0
education         0
education.num     0
marital.status    0
occupation        0
relationship      0
race              0
sex               0
capital.gain      0
capital.loss      0
hours.per.week    0
native.country    0
dtype: int64

### 이상치 처리

In [11]:
train.describe()
# age min : -38

,id,age,fnlwgt,education.num,capital.gain,capital.loss,hours.per.week
count,29304.000000,29304.000000,2.930400e+04,29304.000000,29304.000000,29304.000000,29304.000000
mean,16264.027880,38.552996,1.897488e+05,10.080842,1093.858722,86.744506,40.434036
std,9384.518323,13.626025,1.055250e+05,2.570824,7477.435640,401.518928,12.321306
min,0.000000,-38.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,8145.750000,28.000000,1.177890e+05,9.000000,0.000000,0.000000,40.000000
50%,16253.500000,37.000000,1.783765e+05,10.000000,0.000000,0.000000,40.000000
75%,24374.250000,48.000000,2.370682e+05,12.000000,0.000000,0.000000,45.000000
max,32560.000000,90.000000,1.484705e+06,16.000000,99999.000000,4356.000000,99.000000


In [12]:
print(train.shape)

# 이상치 제거(age > 0 큰 데이터들만 train에 포함)
train = train[train['age']>0]

(29304, 16)


# [문제] 로지스틱회귀 TEST
- 나이(age) - 소득(income) 간의 로지스틱회귀  
- 나이가 증가하면할수록 소득이 높을 확률  

In [16]:
train['income'] = train['income'].replace('<=50K', 0).replace('>50K', 1)
train

/tmp/ipykernel_18355/1300478430.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train['income'] = train['income'].replace('<=50K', 0).replace('>50K', 1)


,id,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,3331,34.0,State-gov,177331,Some-college,10,Married-civ-spouse,Prof-specialty,Husband,Black,Male,4386,0,40.0,United-States,1
1,19749,58.0,Private,290661,HS-grad,9,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,40.0,United-States,0
2,1157,48.0,Private,125933,Some-college,10,Widowed,Exec-managerial,Unmarried,Black,Female,0,1669,38.0,United-States,0
3,693,58.0,Private,100313,Some-college,10,Married-civ-spouse,Protective-serv,Husband,White,Male,0,1902,40.0,United-States,1
4,12522,41.0,Private,195661,Some-college,10,Married-civ-spouse,Transport-moving,Husband,White,Male,0,0,54.0,United-States,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29299,15999,28.0,Private,47168,10th,6,Never-married,Machine-op-inspct,Own-child,White,Female,0,0,40.0,United-States,0
29300,21604,44.0,Local-gov,231793,Doctorate,16,Married-spouse-absent,Prof-specialty,Unmarried,White,Female,0,0,38.0,United-States,0
29301,26839,41.0,Local-gov,201435,HS-grad,9,Married-civ-spouse,Transport-moving,Husband,Black,Male,0,0,40.0,United-States,0
29302,16681,43.0,Private,137722,HS-grad,9,Married-civ-spouse,Handlers-cleaners,Husband,White,Male,0,0,40.0,United-States,0


In [17]:
# 모델 학습
from statsmodels.formula.api import logit
model = logit('income ~ age', data=train).fit()
print(model.summary())

Optimization terminated successfully.
         Current function value: 0.524205
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:                 income   No. Observations:                29301
Model:                          Logit   Df Residuals:                    29299
Method:                           MLE   Df Model:                            1
Date:                Mon, 13 Oct 2025   Pseudo R-squ.:                 0.04923
Time:                        03:28:20   Log-Likelihood:                -15360.
converged:                       True   LL-Null:                       -16155.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -2.7623      0.045    -61.079      0.000      -2.851      -2.674
age            0.0399      0.

In [18]:
# 오즈비값 확인
import numpy as np

# 회귀계수(coef) 확인
print(model.params['age']) 
# 오즈비(Ratio) 확인 (exp(coef))
print(np.exp(model.params['age'])) 

0.03991264299172252
1.0407198560481983


In [19]:
# 소득 확률 확인
X_new = pd.DataFrame([{'age' : 50}])
pred = model.predict(X_new)
pred

0    0.317211
dtype: float64